# 🤖 Intelligent Employee Onboarding Copilot
## Demo Walkthrough — Amiseq AI Engineer Assignment

This notebook demonstrates the full system end-to-end:
1. **RAG Pipeline** — answering new hire questions from onboarding documents
2. **AI Agent** — autonomously executing onboarding tasks
3. **Evaluation Results** — faithfulness and relevance metrics

> **Stack:** LangChain · ChromaDB · Llama3 (Ollama) · sentence-transformers


## 1. Setup & Imports

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv()

print("✅ Environment loaded")
print(f"   LLM Model     : {os.getenv('LLM_MODEL', 'llama3')}")
print(f"   Embedding Model: {os.getenv('EMBEDDING_MODEL', 'all-MiniLM-L6-v2')}")


✅ Environment loaded
   LLM Model     : llama3
   Embedding Model: all-MiniLM-L6-v2


---
## 2. RAG Pipeline — Document Q&A

The RAG pipeline:
1. Loads 5 onboarding documents (Employee Handbook, IT Guide, FAQ, Checklist, Department Docs)
2. Chunks them using `RecursiveCharacterTextSplitter` (500 tokens, 50 overlap)
3. Embeds chunks using `all-MiniLM-L6-v2` and stores in ChromaDB
4. Retrieves top-3 relevant chunks per query
5. Generates grounded answers using Llama3


In [2]:
from src.rag.pipeline import (
    load_documents, chunk_documents,
    create_vectorstore, load_vectorstore,
    build_rag_chain
)
from pathlib import Path

# Load or build vector store
CHROMA_PATH = Path("../data/chroma_db")

if CHROMA_PATH.exists() and any(CHROMA_PATH.iterdir()):
    print("📦 Loading existing vector store...")
    vectorstore = load_vectorstore()
else:
    print("🔨 Building vector store from scratch...")
    docs = load_documents()
    chunks = chunk_documents(docs)
    vectorstore = create_vectorstore(chunks)

rag_chain = build_rag_chain(vectorstore)
print("\n✅ RAG pipeline ready!")


📦 Loading existing vector store...
📦 Loading existing vector store...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🔗 Building RAG chain...
✅ RAG chain ready

✅ RAG pipeline ready!


### 2.1 Test Queries

In [3]:
test_questions = [
    ("Q1", "What is the company's remote work policy?"),
    ("Q2", "How do I set up VPN on my laptop?"),
    ("Q3", "When is the next payroll cycle?"),
    ("Q4", "What's the dress code for the Bangalore office?"),
    ("Q5", "Who is the CEO of Apple?"),  # Out of scope
]

print("=" * 70)
for qid, question in test_questions:
    print(f"\n❓ [{qid}] {question}")
    print("-" * 70)
    answer = rag_chain.invoke(question)
    print(f"💬 {answer}")
    print()



❓ [Q1] What is the company's remote work policy?
----------------------------------------------------------------------
💬 According to the "Amiseq Employee Handbook" (Section 3, Remote Work Policy), Amiseq allows employees to work remotely one day a week. However, this requires prior approval from management and must be done in accordance with our security and data protection policies.

Source: Amiseq Employee Handbook, Section 3, Remote Work Policy


❓ [Q2] How do I set up VPN on my laptop?
----------------------------------------------------------------------
💬 According to the Amiseq Employee Onboarding Guide (p. 12), "To set up VPN on your laptop, follow these steps:

1. Download and install the Cisco AnyConnect VPN client from the company's software portal.
2. Launch the client and enter your Amiseq credentials when prompted.
3. Select the 'Amiseq' VPN connection option.

For more information, refer to the Amiseq IT Department's VPN Setup Guide (available on the company intranet)

---
## 3. AI Agent — Autonomous Onboarding

The Onboarding Agent uses a **ReAct loop** (Reason → Act → Observe):
- Reasons about what needs to be done next
- Calls the appropriate tool
- Observes the result
- Repeats until all tasks are complete

### Available Tools:
| Tool | Purpose |
|---|---|
| `create_it_ticket` | Provision laptop, email, VPN |
| `send_welcome_email` | Send welcome email to new hire |
| `schedule_orientation` | Book orientation session |
| `update_onboarding_status` | Track checklist progress |


In [4]:
from src.agent.agent import OnboardingAgent, onboarding_db
import json

agent = OnboardingAgent()

task = """
Onboard Priya Sharma. She is joining the Engineering team in the 
Bangalore office on May 5, 2026. Her email is priya.sharma@amiseq.com.
Make sure her laptop is provisioned, she gets a welcome email, 
orientation is scheduled, and her onboarding checklist is started.
"""

result = agent.run(task)



🤖 Agent Starting Task:
📋 
Onboard Priya Sharma. She is joining the Engineering team in the 
Bangalore office on May 5, 2026. Her email is priya.sharma@amiseq.com.
Make sure her laptop is provisioned, she gets a welcome email, 
orientation is scheduled, and her onboarding checklist is started.

🔑 Run ID: 8e1e8e6c-1942-43fe-af19-5d50704f50a9

🔄 Step 1:
  💭 Thought: Start the onboarding process by provisioning Priya's laptop and other necessary tools
  🔧 Action: create_it_ticket({'employee_name': 'Priya Sharma', 'request_type': 'new hire'})
  🖥️  IT Ticket: IT ticket IT-20260415135102 created for Priya Sharma — new hire provisioning started.

🔄 Step 2:
  💭 Thought: Next step is to send a welcome email to Priya Sharma
  🔧 Action: send_welcome_email({'employee_name': 'Priya Sharma', 'email': 'priya.sharma@amiseq.com', 'start_date': 'May 5, 2026'})
  📧 Email: Welcome email sent to Priya Sharma at priya.sharma@amiseq.com for start date May 5, 2026.

🔄 Step 3:
  💭 Thought: Next step is to sch

### 3.1 Onboarding Database State

In [5]:
print("\n📦 Onboarding Database State:")
print(json.dumps(onboarding_db, indent=2))



📦 Onboarding Database State:
{
  "": {
    "onboarding": {
      "status": "completed",
      "updated_at": "2026-04-15T13:51:36.877065"
    }
  }
}


---
## 4. Evaluation Results

The evaluation module measures RAG quality using two **RAGAS-style metrics**:

- **Faithfulness** — Are all claims in the answer grounded in retrieved context? (0-1)
- **Answer Relevance** — Does the answer address the question asked? (0-1)
- **Overall Score** — Harmonic mean of both metrics
- **Hallucination Risk** — Flagged if either metric < 0.5

All scoring uses **LLM-as-judge** via Llama3 locally.


In [6]:
import json
from pathlib import Path

results_path = Path("../logs/eval_results.json")

with open(results_path) as f:
    eval_data = json.load(f)

summary = eval_data["summary"]
samples = eval_data["samples"]

print("=" * 70)
print("📈  RAG PIPELINE EVALUATION REPORT")
print("=" * 70)
print(f"  Samples Evaluated    : {summary['n_samples']}")
print(f"  Avg Faithfulness     : {summary['avg_faithfulness']:.3f} / 1.000")
print(f"  Avg Answer Relevance : {summary['avg_answer_relevance']:.3f} / 1.000")
print(f"  Avg Overall Score    : {summary['avg_overall_score']:.3f} / 1.000")
print(f"  Hallucination Risks  : {summary['hallucination_risk_count']} / {summary['n_samples']} samples")
print(f"  Avg Eval Latency     : {summary['avg_eval_latency_ms']:.0f} ms / sample")
print("-" * 70)


📈  RAG PIPELINE EVALUATION REPORT
  Samples Evaluated    : 10
  Avg Faithfulness     : 1.000 / 1.000
  Avg Answer Relevance : 0.830 / 1.000
  Avg Overall Score    : 0.856 / 1.000
  Hallucination Risks  : 1 / 10 samples
  Avg Eval Latency     : 7422 ms / sample
----------------------------------------------------------------------


### 4.1 Per-Sample Results

In [7]:
print(f"  {'ID':<5} {'Category':<14} {'Faith':>6} {'Relev':>6} {'Overall':>8} {'Risk'}")
print(f"  {'-'*5} {'-'*14} {'-'*6} {'-'*6} {'-'*8} {'-'*8}")

for r in samples:
    flag = "🚨 RISK" if r["hallucination_risk"] else "✅ OK"
    print(
        f"  {r['id']:<5} {r['category']:<14} "
        f"{r['faithfulness']:>6.2f} {r['answer_relevance']:>6.2f} "
        f"{r['overall_score']:>8.2f} {flag}"
    )

print()
print("Note: Q09 (out-of-scope) is correctly declined by the system.")
print("The hallucination flag is a known false positive for out-of-scope rejection.")


  ID    Category        Faith  Relev  Overall Risk
  ----- -------------- ------ ------ -------- --------
  Q01   policy           1.00   1.00     1.00 ✅ OK
  Q02   it_setup         1.00   1.00     1.00 ✅ OK
  Q03   payroll          1.00   1.00     1.00 ✅ OK
  Q04   policy           1.00   1.00     1.00 ✅ OK
  Q05   leave            1.00   1.00     1.00 ✅ OK
  Q06   it_setup         1.00   0.50     0.67 ✅ OK
  Q07   onboarding       1.00   1.00     1.00 ✅ OK
  Q08   expenses         1.00   0.80     0.89 ✅ OK
  Q09   out_of_scope     1.00   0.00     0.00 🚨 RISK
  Q10   department       1.00   1.00     1.00 ✅ OK

Note: Q09 (out-of-scope) is correctly declined by the system.
The hallucination flag is a known false positive for out-of-scope rejection.


---
## 5. Summary

| Component | Status | Key Metric |
|---|---|---|
| RAG Pipeline | ✅ Working | Faithfulness: 1.000 |
| AI Agent | ✅ Working | All 4 tools executed |
| Evaluation | ✅ Working | Overall Score: 0.856 |
| Hallucination Handling | ✅ Working | Out-of-scope correctly declined |

### Production Improvements
- Replace Llama3 with GPT-4o or Claude for better instruction following
- Migrate ChromaDB to Pinecone for scalability
- Replace mocked tools with real API integrations (Jira, SendGrid, Google Calendar, HiBob)
- Add semantic caching to reduce latency
- Integrate LangSmith for full distributed tracing
